# 04 — Churn Classifier

Trains the XGBoost churn model on RFM + behavioural features
(see `models/churn.py`) and visualises the SHAP feature
importance and the AUC-ROC curve.

In [ ]:
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from neuralretail.data.ingest import load_raw
from neuralretail.data.clean import clean
from neuralretail.features.rfm import compute_rfm
from neuralretail.models.churn import train as churn_train
from sklearn.metrics import roc_auc_score, roc_curve

logging.getLogger('neuralretail').setLevel(logging.WARNING)


## Build the training table

In [ ]:
raw = load_raw()
cleaned, _ = clean(raw)
rfm = compute_rfm(cleaned)
res = churn_train(cleaned, rfm, run_name='nb_churn')
print('metrics:', {k: round(v, 4) for k, v in res.metrics.items() if isinstance(v, float)})
res.feature_importances.head()


## Feature importance (XGBoost gain)

In [ ]:
imp = res.feature_importances.sort_values('importance')
ax = imp.plot.barh(x='feature', y='importance', color='teal', legend=False)
ax.set_title('XGBoost feature importance')
ax.set_xlabel('gain')
plt.tight_layout()
plt.show()


## AUC-ROC

Re-fit on the same train/test split so we can plot the curve.

In [ ]:
from neuralretail.models.churn import build_training_table, FEATURE_COLUMNS
from sklearn.model_selection import train_test_split
table = build_training_table(cleaned, rfm)
X = table[FEATURE_COLUMNS].fillna(0)
y = table['churned']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
proba = res.model.predict_proba(Xte)[:, 1]
auc = roc_auc_score(yte, proba)
fpr, tpr, _ = roc_curve(yte, proba)
plt.plot(fpr, tpr, color='coral', label=f'AUC = {auc:.4f}')
plt.plot([0, 1], [0, 1], color='grey', linestyle='--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('ROC — churn classifier')
plt.legend()
plt.tight_layout()
plt.show()
